# Data Loading

Data link:

https://www.phoenixopendata.com/dataset/officer-show-of-force/resource/7e9d5fc7-ce02-4108-80af-369b1b54c4ff

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import psycopg2
from dotenv import load_dotenv
import os

We'll load the data from the API and then save it to a CSV. If the CSV already exists, we can just use that.

In [2]:
batch_size = 1000

# URL and parameters for the request
url = 'https://www.phoenixopendata.com/api/3/action/datastore_search'
params = {
    'resource_id' : '7e9d5fc7-ce02-4108-80af-369b1b54c4ff',
    'limit' : batch_size,
    'offset' : 0
}

# List holding the data
data = []

# The size of the last batch (intitial value of 1 to ensure the loop runs)
last_batch_size = 1

# Check if the CSV file already exists (if so, we can just use that instead of batch loading)
if os.path.exists('data_raw.csv'):
    print('CSV file found - loading data from file')
    df = pd.read_csv('data_raw.csv',na_values=[""],keep_default_na=False)
else:
    print('CSV file NOT found - loading data from API')

    # While there are still more rows to load
    while last_batch_size > 0:
        # Make request
        response = requests.get(url, params=params)
        response.raise_for_status()

        # Get batch from response JSON and add it to the overall data list
        current_batch = response.json()['result']['records']
        data.extend(current_batch)

        # Move up the offset based on the batch size
        params['offset'] = params['offset'] + batch_size

        # Update last batch size
        last_batch_size = len(current_batch)
        print(f'Loaded batch with size {last_batch_size}')

    # Convert to pandas dataframe and save to csv
    df = pd.DataFrame(data)
    df.to_csv('data_raw.csv',index=False)

print('Done')

CSV file found - loading data from file
Done


In [3]:
df.head()

,_id,INC_IR_NO,INC_IA_NO,INC_DATE,INC_YEAR,INC_TIME,INC_DAY_WEEK,INC_BEAT,HUNDRED_BLOCK,INC_CITY,...,CIT_NUMBER,CIT_GENDER,CIT_AGE,SUBJ_AGE_GROUP,CIT_RACE,CIT_ETHNICITY,SIMPLE_SUBJ_RE_GRP,CITIZEN_CHARGE,HIGHEST_SHOW_FORCE,SHOW_FORCE_COUNT
0,1,2.025000e+14,SOF25-0005,2025-02-18T00:00:00,2025,23:11,Tuesday,Maricopa,1XXXX West Rancho Santa Fe Boulevard,Avondale,...,44041.0,Male,30,30s,Black,Non-Hispanic,Black or African American,None,Gun,1
1,2,2.025000e+14,SOF25-0006,2025-02-18T00:00:00,2025,13:50,Tuesday,721 Beat,6XXX West Mcdowell Road,Phoenix,...,61390.0,Male,29,20s,White,Hispanic / Latino,White,None,Gun,1
2,3,2.025000e+14,SOF25-0007,2025-02-18T00:00:00,2025,11:13,Tuesday,731 Beat,2XXX West Heatherbrae Drive,Phoenix,...,61392.0,Male,45,40s,White,Not Hispanic / Latino,White,None,Gun,1
3,4,2.025000e+14,SOF25-0010,2025-02-18T00:00:00,2025,12:44,Tuesday,213 Beat,4XXX East Maya Way,Cave Creek,...,61423.0,Male,53,50s,White,Hispanic / Latino,White,Criminal Felony,Gun,1
4,5,2.025000e+14,SOF25-0023,2025-02-18T00:00:00,2025,12:45,Tuesday,431 Beat,4XXX South 32nd Street,Phoenix,...,61465.0,Male,19,<20,Black / African American,Not Hispanic / Latino,Black or African American,Criminal Felony,Gun,1


In [4]:
df.iloc[0]

_id                                                      1
INC_IR_NO                                202500000255424.0
INC_IA_NO                                       SOF25-0005
INC_DATE                               2025-02-18T00:00:00
INC_YEAR                                              2025
INC_TIME                                             23:11
INC_DAY_WEEK                                       Tuesday
INC_BEAT                                          Maricopa
HUNDRED_BLOCK         1XXXX West Rancho Santa Fe Boulevard
INC_CITY                                          Avondale
INC_STATE                                               AZ
INC_ZIPCODE                                          85392
INC_PRECINCT                                   Out of City
CIT_NUMBER                                         44041.0
CIT_GENDER                                            Male
CIT_AGE                                                 30
SUBJ_AGE_GROUP                                         3

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4730 entries, 0 to 4729
Data columns (total 23 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   _id                 4730 non-null   int64  
 1   INC_IR_NO           4727 non-null   float64
 2   INC_IA_NO           4730 non-null   object 
 3   INC_DATE            4730 non-null   object 
 4   INC_YEAR            4730 non-null   int64  
 5   INC_TIME            4726 non-null   object 
 6   INC_DAY_WEEK        4730 non-null   object 
 7   INC_BEAT            4730 non-null   object 
 8   HUNDRED_BLOCK       4730 non-null   object 
 9   INC_CITY            4730 non-null   object 
 10  INC_STATE           4730 non-null   object 
 11  INC_ZIPCODE         4730 non-null   object 
 12  INC_PRECINCT        4730 non-null   object 
 13  CIT_NUMBER          4728 non-null   float64
 14  CIT_GENDER          4730 non-null   object 
 15  CIT_AGE             4730 non-null   object 
 16  SUBJ_A

In [6]:
df.describe()

,_id,INC_IR_NO,INC_YEAR,CIT_NUMBER,SHOW_FORCE_COUNT
count,4730.000000,4.727000e+03,4730.0,4728.000000,4730.0
mean,2365.500000,1.643595e+14,2025.0,63586.128173,1.0
std,1365.577717,1.045989e+14,0.0,11317.278128,0.0
min,1.000000,2.025133e+10,2025.0,22.000000,1.0
25%,1183.250000,2.025000e+14,2025.0,62752.500000,1.0
50%,2365.500000,2.025000e+14,2025.0,66396.000000,1.0
75%,3547.750000,2.025000e+14,2025.0,70056.750000,1.0
max,4730.000000,2.025000e+15,2025.0,75576.000000,1.0


# Data Cleaning

Checking for duplicates and NA values:

In [7]:
print(f'Number of duplicates: {df.duplicated().sum()}')

Number of duplicates: 0


In [8]:
print(f'Number of rows with NA values: {df.isna().any(axis=1).sum()}')

Number of rows with NA values: 9


In [9]:
df[df.isna().any(axis=1)]

,_id,INC_IR_NO,INC_IA_NO,INC_DATE,INC_YEAR,INC_TIME,INC_DAY_WEEK,INC_BEAT,HUNDRED_BLOCK,INC_CITY,...,CIT_NUMBER,CIT_GENDER,CIT_AGE,SUBJ_AGE_GROUP,CIT_RACE,CIT_ETHNICITY,SIMPLE_SUBJ_RE_GRP,CITIZEN_CHARGE,HIGHEST_SHOW_FORCE,SHOW_FORCE_COUNT
2530,2531,2.025000e+14,SOF25-2091,2025-07-19T00:00:00,2025,NaN,Saturday,512 Beat,1XX West Washington Street,Phoenix,...,68361.0,Male,43,40s,Black / African American,Not Hispanic / Latino,Black or African American,Criminal Misdemeanor,Taser,1
3698,3699,NaN,SOF25-2843,2025-10-03T00:00:00,2025,20:50,Friday,231 Beat,2XXX East Vista Drive,Phoenix,...,70974.0,Female,38,30s,White,Not Hispanic / Latino,White,Criminal Misdemeanor,Gun,1
4094,4095,2.025016e+11,SOF25-3194,2025-11-04T00:00:00,2025,NaN,Tuesday,911 Beat,4XXX North 27th Avenue,Phoenix,...,72078.0,Male,23,20s,White,Not Hispanic / Latino,White,None,Gun,1
4341,4342,2.025017e+11,SOF25-3475,2025-11-25T00:00:00,2025,00:02,Tuesday,621 Beat,2XXX West Corrine Drive,Phoenix,...,NaN,Not Available,Not Available,Not Available,Not Available,Not Available,Other,None,Impact Munitions,1
4461,4462,NaN,SOF25-3539,2025-12-06T00:00:00,2025,16:49,Saturday,413 Beat,5XXX South 35th Avenue,Phoenix,...,73225.0,Male,27,20s,Black / African American,Not Hispanic / Latino,Black or African American,Criminal Felony,Gun,1
4521,4522,2.025018e+11,SOF25-3563,2025-12-13T00:00:00,2025,00:27,Saturday,624 Beat,1XXX Las Palmaritas Drive,Phoenix,...,NaN,Not Available,Not Available,Not Available,Not Available,Not Available,Other,Criminal Felony,None,1
4614,4615,NaN,SOF25-3636,2025-12-20T00:00:00,2025,01:45,Saturday,821 Beat,5XXX West Lewis Avenue,Phoenix,...,71831.0,Male,56,50s,White,Hispanic / Latino,White,None,Gun,1
4617,4618,2.025018e+11,SOF25-3645,2025-12-20T00:00:00,2025,NaN,Saturday,221 Beat,1XXXX North 27th Drive,Phoenix,...,48962.0,Male,17,<20,Black,Non-Hispanic,Black or African American,None,Gun,1
4722,4723,2.025019e+11,SOF26-0037,2025-12-31T00:00:00,2025,NaN,Wednesday,413 Beat,1XXX West Chipman Road,Phoenix,...,73735.0,Male,25,20s,White,Hispanic / Latino,White,Criminal Felony,Gun,1


There are a few NA values, but no duplicate rows. We'll go through all of the columns, making sure that the types are correct and that any duplicate/null values are taken care of. Of course, how we handle the duplicates/null values (if at all) will depend on the column.

Before we get started, I noticed that the dataset often uses 'Not Available' to denote NA values. For consistency, let's just replace these instances with ```pd.NA```:

In [10]:
df.replace('Not Available', pd.NA, inplace=True)

Now we can clean each column one by one.

## Cleaning ```_id```

In [11]:
df['_id'].unique()

array([   1,    2,    3, ..., 4728, 4729, 4730], dtype=int64)

In [12]:
print(f"Number of duplicates: {df['_id'].duplicated().sum()}")

Number of duplicates: 0


In [13]:
print(f"Number of NA values: {df['_id'].isna().sum()}")

Number of NA values: 0


Everything looks good already.

## Cleaning ```INC_IR_NO```

In [14]:
df['INC_IR_NO'].unique()

array([2.02500000e+14, 2.02500000e+14, 2.02500000e+14, ...,
       2.02501868e+11, 2.02501868e+11, 2.02501862e+11])

We'll need to convert these strings to ints.

In [15]:
print(f"Number of duplicates: {df['INC_IR_NO'].duplicated().sum()}")

Number of duplicates: 1239


There are quite a lot of duplicates. Let's see what's going on here:

In [16]:
df[df['INC_IR_NO'].duplicated()].head()

,_id,INC_IR_NO,INC_IA_NO,INC_DATE,INC_YEAR,INC_TIME,INC_DAY_WEEK,INC_BEAT,HUNDRED_BLOCK,INC_CITY,...,CIT_NUMBER,CIT_GENDER,CIT_AGE,SUBJ_AGE_GROUP,CIT_RACE,CIT_ETHNICITY,SIMPLE_SUBJ_RE_GRP,CITIZEN_CHARGE,HIGHEST_SHOW_FORCE,SHOW_FORCE_COUNT
6,7,2.025000e+14,SOF25-0027,2025-02-18T00:00:00,2025,17:15,Tuesday,621 Beat,1XXXX North 28th Drive,Phoenix,...,61478.0,Male,23,20s,White,Hispanic / Latino,White,None,Gun,1
16,17,2.025000e+14,SOF25-0009,2025-02-19T00:00:00,2025,19:11,Wednesday,921 Beat,2XXX West Northern Avenue,Phoenix,...,45281.0,Male,28,20s,Black,Non-Hispanic,Black or African American,None,Irritants,1
17,18,2.025000e+14,SOF25-0009,2025-02-19T00:00:00,2025,19:11,Wednesday,921 Beat,2XXX West Northern Avenue,Phoenix,...,61419.0,Female,32,30s,White,Not Hispanic / Latino,White,None,Irritants,1
18,19,2.025000e+14,SOF25-0009,2025-02-19T00:00:00,2025,19:11,Wednesday,921 Beat,2XXX West Northern Avenue,Phoenix,...,61421.0,Male,42,40s,White,Not Hispanic / Latino,White,None,Irritants,1
26,27,2.025000e+14,SOF25-0017,2025-02-19T00:00:00,2025,15:33,Wednesday,515 Beat,1XX West Papago Street,Phoenix,...,61446.0,Male,28,20s,White,Hispanic / Latino,White,Criminal Felony,Gun,1


It appears that one incident report number (INC_IR_NO) can have multiple citizens (CIT_GENDER), which is why there are so many duplicate report numbers. Since this is a normalization issue, we'll handle in the normalization section later.

In [17]:
print(f"Number of NA values: {df['INC_IR_NO'].isna().sum()}")

Number of NA values: 3


We just need to convert the row to numeric format:

In [18]:
df['INC_IR_NO'] = df['INC_IR_NO'].astype('Int64')

In [19]:
df['INC_IR_NO'].unique()

<IntegerArray>
[202500000255424, 202500000193276, 202500000252010, 202500000252106,
 202500000189932, 202500000087063, 202500000252086, 202500000253079,
 202500000252554, 202500000250443,
 ...
    202501862789,    202501865482,    202501864350,    202501866002,
    202501868518,    202501757380,    202501862691,    202501868103,
    202501867554,    202501861673]
Length: 3491, dtype: Int64

## Cleaning ```INC_DATE```

In [20]:
df['INC_DATE'].unique()[:10]

array(['2025-02-18T00:00:00', '2025-02-19T00:00:00',
       '2025-02-20T00:00:00', '2025-02-21T00:00:00',
       '2025-02-22T00:00:00', '2025-02-23T00:00:00',
       '2025-02-24T00:00:00', '2025-02-25T00:00:00',
       '2025-02-26T00:00:00', '2025-02-27T00:00:00'], dtype=object)

In [21]:
print(f"Number of duplicates: {df['INC_DATE'].duplicated().sum()}")
print(f"Number of NA values: {df['INC_DATE'].isna().sum()}")

Number of duplicates: 4413
Number of NA values: 0


Duplicates are to be expected, since multiple incidents can happen on the same day.

We just need to convert the dates from strings to actual date objects. Since pandas ```datetime``` objects store the time as well, we'll fill in the time using the values in the ```INC_TIME``` column.

In [22]:
def fix_INC_DATE(row):
  INC_DATE = row['INC_DATE']
  INC_TIME = row['INC_TIME']

  # If date is NA, do nothing
  if pd.isna(INC_DATE):
    return pd.NA

  # If INC_TIME exists
  if not pd.isna(INC_TIME):
    # Remove "T00:00:00" substring
    INC_DATE = INC_DATE.replace('T00:00:00', '')

    # Add the time from the INC_TIME column
    INC_DATE = INC_DATE + 'T' + INC_TIME + ':00'

  # Convert to date
  INC_DATE = pd.to_datetime(INC_DATE)

  return INC_DATE

In [23]:
df['INC_DATE'] = df.apply(fix_INC_DATE, axis=1)

In [24]:
df['INC_DATE'].unique()[:10]

<DatetimeArray>
['2025-02-18 23:11:00', '2025-02-18 13:50:00', '2025-02-18 11:13:00',
 '2025-02-18 12:44:00', '2025-02-18 12:45:00', '2025-02-18 17:15:00',
 '2025-02-18 11:30:00', '2025-02-18 15:05:00', '2025-02-18 15:04:00',
 '2025-02-18 01:28:00']
Length: 10, dtype: datetime64[ns]

## Cleaning ```INC_YEAR```

In [25]:
df['INC_YEAR'].unique()

array([2025], dtype=int64)

There is just one year, so let's convert it to an integer:

In [26]:
df['INC_YEAR'] = pd.to_numeric(df['INC_YEAR'])

## Cleaning ```INC_TIME```

In [27]:
df['INC_TIME'].unique()

array(['23:11', '13:50', '11:13', ..., '11:44', '11:27', '13:16'],
      dtype=object)

In [28]:
print(f"Number of duplicates: {df['INC_TIME'].duplicated().sum()}")
print(f"Number of NA values: {df['INC_TIME'].isna().sum()}")

Number of duplicates: 3547
Number of NA values: 4


There are a few duplicates, which make sense since a single incident report can have multiple rows. As for NA values, we'll just have to fill in a default value.

Since we already cleaned the date, we can just extract the time component for ```INC_DATE```:

In [29]:
def fix_INC_TIME(row):
  if pd.isna(row['INC_DATE']):
    return pd.NA

  # Extract the time from INC_DATE and return it
  return row['INC_DATE'].time()

In [30]:
df['INC_TIME'] = df.apply(fix_INC_TIME, axis=1)

In [31]:
df['INC_TIME'].unique()

array([datetime.time(23, 11), datetime.time(13, 50),
       datetime.time(11, 13), ..., datetime.time(11, 44),
       datetime.time(11, 27), datetime.time(13, 16)], dtype=object)

## Cleaning ```INC_DAY_WEEK```

In [32]:
df['INC_DAY_WEEK'].unique()

array(['Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday',
       'Monday'], dtype=object)

In [33]:
print(f"Number of duplicates: {df['INC_DAY_WEEK'].duplicated().sum()}")
print(f"Number of NA values: {df['INC_DAY_WEEK'].isna().sum()}")

Number of duplicates: 4723
Number of NA values: 0


Everything looks good here.

## Cleaning ```INC_BEAT```

In [34]:
df['INC_BEAT'].unique()

array(['Maricopa', '721 Beat', '731 Beat', '213 Beat', '431 Beat',
       '621 Beat', '822 Beat', '831 Beat', '812 Beat', '413 Beat',
       '824 Beat', '423 Beat', '921 Beat', '924 Beat', '232 Beat',
       '223 Beat', '736 Beat', '515 Beat', '711 Beat', '734 Beat',
       '513 Beat', '224 Beat', '813 Beat', '825 Beat', '414 Beat',
       '421 Beat', '412 Beat', '923 Beat', '913 Beat', '911 Beat',
       '723 Beat', '511 Beat', '814', '832 Beat', '821 Beat', '622 Beat',
       '922 Beat', '713 Beat', '726 Beat', '814 Beat', '725 Beat',
       '411 Beat', '932 Beat', '222 Beat', '724 Beat', '823 Beat',
       '625 Beat', '611 Beat', '912 Beat', '424 Beat', '422 Beat',
       '833 Beat', '613 Beat', '634 Beat', '735 Beat', '915 Beat',
       '221 Beat', '834 Beat', '732 Beat', '914 Beat', '722 Beat',
       '231 Beat', '514 Beat', '815 Beat', '233 Beat', '934 Beat',
       '624 Beat', '822', '733 Beat', '933 Beat', '925 Beat', '623 Beat',
       '612 Beat', '614 Beat', '931 Beat', '811 

In [35]:
print(f"Number of duplicates: {df['INC_BEAT'].duplicated().sum()}")
print(f"Number of NA values: {df['INC_BEAT'].isna().sum()}")

Number of duplicates: 4607
Number of NA values: 5


For consistency, we'll remove the 'Beat' substring from all of the values. This is because some include it and others don't, which is why it's best to just remove it for consistency.

In [36]:
def fix_INC_BEAT(row):
  INC_BEAT = row['INC_BEAT']

  # For NA values, do nothing
  if pd.isna(INC_BEAT):
    return INC_BEAT

  # Remove `Beat` substring
  INC_BEAT = INC_BEAT.replace('Beat', '')

  # Also remove `(Airport)` substring
  INC_BEAT = INC_BEAT.replace('(Airport)', '')

  # Strip string
  INC_BEAT = INC_BEAT.strip()

  return INC_BEAT

In [37]:
df['INC_BEAT'] = df.apply(fix_INC_BEAT, axis=1)

In [38]:
df['INC_BEAT'].unique()

array(['Maricopa', '721', '731', '213', '431', '621', '822', '831', '812',
       '413', '824', '423', '921', '924', '232', '223', '736', '515',
       '711', '734', '513', '224', '813', '825', '414', '421', '412',
       '923', '913', '911', '723', '511', '814', '832', '821', '622',
       '922', '713', '726', '725', '411', '932', '222', '724', '823',
       '625', '611', '912', '424', '422', '833', '613', '634', '735',
       '915', '221', '834', '732', '914', '722', '231', '514', '815',
       '233', '934', '624', '733', '933', '925', '623', '612', '614',
       '931', '811', '615', '712', '234', '211', '425', '516', '434',
       '631', '512', '214', '432', '714', '632', '435', '633', '715',
       '591', '433', '212', <NA>, 'Pinal'], dtype=object)

## Cleaning ```HUNDRED_BLOCK```

In [39]:
df['HUNDRED_BLOCK'].unique()

array(['1XXXX West Rancho Santa Fe Boulevard', '6XXX West Mcdowell Road',
       '2XXX West Heatherbrae Drive', ..., '1XXX West Chipman Road',
       '2XX East Clarendon Avenue', '1XXX North 3rd Street'], dtype=object)

In [40]:
print(f"Number of duplicates: {df['HUNDRED_BLOCK'].duplicated().sum()}")
print(f"Number of NA values: {df['HUNDRED_BLOCK'].isna().sum()}")

Number of duplicates: 2827
Number of NA values: 236


Everything looks good here.

## Cleaning ```INC_CITY```

In [41]:
df['INC_CITY'].unique()

array(['Avondale', 'Phoenix', 'Cave Creek', 'Tolleson', 'Mesa',
       'Goodyear', <NA>, 'Chandler', 'Glendale', 'Laveen', 'Peoria',
       'Tempe', 'Gilbert', 'Scottsdale', 'Surprise', 'Buckeye',
       'New River', 'Guadalupe', 'Carefree', 'Maricopa',
       'Litchfield Park', 'El Mirage', 'Apache Junction',
       'Fountain Hills', 'Anthem', 'Queen Creek', 'Maricopa Colony',
       'Casa Blanca'], dtype=object)

In [42]:
print(f"Number of duplicates: {df['INC_CITY'].duplicated().sum()}")
print(f"Number of NA values: {df['INC_CITY'].isna().sum()}")

Number of duplicates: 4702
Number of NA values: 245


## Cleaning ```INC_STATE```

In [43]:
df['INC_STATE'].unique()

array(['AZ', <NA>], dtype=object)

In [44]:
print(f"Number of duplicates: {df['INC_STATE'].duplicated().sum()}")
print(f"Number of NA values: {df['INC_STATE'].isna().sum()}")

Number of duplicates: 4728
Number of NA values: 245


We have some NA values, but since this is data for Phoenix, Arizona and the surrounding areas, let's just assume that all incidents are in AZ.

In [45]:
df['INC_STATE'] = 'AZ'

In [46]:
df['INC_STATE'].unique()

array(['AZ'], dtype=object)

## Cleaning ```INC_ZIPCODE```

In [47]:
df['INC_ZIPCODE'].unique()

array(['85392', '85035', '85015', '85331', '85040', '85029', '85031',
       '85043', '85037', '85041', <NA>, '85210', '85034', '85032',
       '85301', '85022', '85006', '85003', '85016', '85007', '85027',
       '85033', '85009', '85395', '85017', '85051', '85019', '85018',
       '85020', '85224', '85013', '85204', '85008', '85306', '85014',
       '85021', '85042', '85339', '85028', '85023', '85053', '85307',
       '85083', '85004', '85050', '85085', '85254', '85012', '85308',
       '85044', '85381', '85283', '85024', '85045', '85086', '85251',
       '85304', '85233', '85353', '85303', '85345', '85351', '85323',
       '85259', '85302', '85374', '85286', '85326', '85203', '85087',
       '85310', '85377', '85281', '85256', '85138', '85048', '85250',
       '85340', '85282', '85305', '85226', '85288', '85201', '85206',
       '85257', '85396', '85284', '85338', '85255', '85379', '85335',
       '85119', '85011', '85329', '85383', '85382', '85144', '85253',
       '85054', '85296'

In [48]:
print(f"Number of duplicates: {df['INC_ZIPCODE'].duplicated().sum()}")
print(f"Number of NA values: {df['INC_ZIPCODE'].isna().sum()}")

Number of duplicates: 4629
Number of NA values: 260


We just need to convert the column to numerical values:

In [49]:
df['INC_ZIPCODE'] = pd.to_numeric(df['INC_ZIPCODE'])

## Cleaning ```INC_PRECINCT```

In [50]:
df['INC_PRECINCT'].unique()

array(['Out of City', 'Maryvale/Estrella Precinct',
       'Mountain View Precinct', 'Black Mountain Precinct',
       'South Mountain Precinct', 'Cactus Park Precinct',
       'Desert Horizon Precinct', 'Central City Precinct', <NA>],
      dtype=object)

In [51]:
print(f"Number of duplicates: {df['INC_PRECINCT'].duplicated().sum()}")
print(f"Number of NA values: {df['INC_PRECINCT'].isna().sum()}")

Number of duplicates: 4721
Number of NA values: 105


Everything looks good here.

## Cleaning ```CIT_NUMBER```

In [52]:
df['CIT_NUMBER'].unique()

array([44041., 61390., 61392., ..., 74248., 74261., 74267.])

In [53]:
print(f"Number of duplicates: {df['CIT_NUMBER'].duplicated().sum()}")
print(f"Number of NA values: {df['CIT_NUMBER'].isna().sum()}")

Number of duplicates: 183
Number of NA values: 2


Everything looks good here. The NA values might prove troublesome if we use this as a primary key for a citizens table, but we'll leave that for the normalization section. Let's just convert the column to numeric format (if it isn't already):

In [54]:
df['CIT_NUMBER'] = pd.to_numeric(df['CIT_NUMBER'])

## Cleaning ```CIT_GENDER```

In [55]:
df['CIT_GENDER'].unique()

array(['Male', 'Female', <NA>], dtype=object)

Everything looks good here.

## Cleaning ```CIT_AGE```

In [56]:
df['CIT_AGE'].unique()

array(['30', '29', '45', '53', '19', '23', '38', '43', '33', '34', '21',
       '42', '35', '28', '32', '54', '20', '49', '0', '36', '26', '47',
       '41', '44', '31', '65', '37', '50', '48', '55', '46', '15', '71',
       '60', '27', <NA>, '39', '25', '40', '63', '5', '3', '1', '16',
       '58', '59', '13', '24', '56', '51', '22', '77', '66', '18', '17',
       '67', '57', '69', '14', '52', '62', '7', '75', '76', '61', '72',
       '64', '68', '81', '70', '78', '12', '9', '79', '74', '82', '83'],
      dtype=object)

We can convert this column to integers. To keep this future proof, we'll also check if the age is invalid (below 0 or over 100) just to ensure that no invalid ages are added later on.

In [57]:
df['CIT_AGE'] = pd.to_numeric(df['CIT_AGE'])

In [58]:
def fix_CIT_AGE(row):
  CIT_AGE = row['CIT_AGE']

  # For 'Not Available' return NA
  if CIT_AGE == 'Not Available':
    return pd.NA

  # If age is < 0 or > 100, return NA
  if CIT_AGE < 0 or CIT_AGE > 100:
    return pd.NA

  return CIT_AGE

In [59]:
df['CIT_AGE'] = df.apply(fix_CIT_AGE, axis=1)

In [60]:
df['CIT_AGE'].unique()

array([30., 29., 45., 53., 19., 23., 38., 43., 33., 34., 21., 42., 35.,
       28., 32., 54., 20., 49.,  0., 36., 26., 47., 41., 44., 31., 65.,
       37., 50., 48., 55., 46., 15., 71., 60., 27., nan, 39., 25., 40.,
       63.,  5.,  3.,  1., 16., 58., 59., 13., 24., 56., 51., 22., 77.,
       66., 18., 17., 67., 57., 69., 14., 52., 62.,  7., 75., 76., 61.,
       72., 64., 68., 81., 70., 78., 12.,  9., 79., 74., 82., 83.])

## Cleaning ```SUBJ_AGE_GROUP```

In [61]:
df['SUBJ_AGE_GROUP'].unique()

array(['30s', '20s', '40s', '50s', '<20', '60s', '70s', <NA>, '80s'],
      dtype=object)

Since we cleaned the ages already, we can just use those to determine the age group:

In [62]:
def fix_SUBJ_AGE_GROUP(row):
  CIT_AGE = row['CIT_AGE']

  # If age is NA, return NA
  if pd.isna(CIT_AGE):
    return pd.NA
  elif CIT_AGE < 20:
    return '<20'
  elif CIT_AGE < 30:
    return '20s'
  elif CIT_AGE < 40:
    return '30s'
  elif CIT_AGE < 50:
    return '40s'
  elif CIT_AGE < 60:
    return '50s'
  elif CIT_AGE < 70:
    return '60s'
  elif CIT_AGE < 80:
    return '70s'
  elif CIT_AGE < 90:
    return '80s'

  return CIT_AGE

In [63]:
df['SUBJ_AGE_GROUP'] = df.apply(fix_SUBJ_AGE_GROUP, axis=1)

In [64]:
df['SUBJ_AGE_GROUP'].unique()

array(['30s', '20s', '40s', '50s', '<20', '60s', '70s', <NA>, '80s'],
      dtype=object)

## Cleaning ```CIT_RACE```

In [65]:
df['CIT_RACE'].unique()

array(['Black', 'White', 'Black / African American', <NA>,
       'American Indian / Alaskan Native', 'Unknown', 'Asian',
       'Native Hawaiian / Other Pacific Islander', 'white', 'Hispanic',
       'Asian / Pacific Islander', 'whi'], dtype=object)

We just need to remove redundant values:

In [66]:
def fix_CIT_RACE(row):
  CIT_RACE = row['CIT_RACE']

  # Do nothing for NA values
  if pd.isna(CIT_RACE):
    return CIT_RACE

  if 'whi' in CIT_RACE.lower():
    return 'White'
  elif 'black' in CIT_RACE.lower():
    return 'Black'
  elif 'asian' in CIT_RACE.lower():
    return 'Asian / Pacific Islander'
  elif 'unknown' in CIT_RACE.lower():
    return pd.NA

  return CIT_RACE

In [67]:
df['CIT_RACE'] = df.apply(fix_CIT_RACE, axis=1)

In [68]:
df['CIT_RACE'].unique()

array(['Black', 'White', <NA>, 'American Indian / Alaskan Native',
       'Asian / Pacific Islander',
       'Native Hawaiian / Other Pacific Islander', 'Hispanic'],
      dtype=object)

## Cleaning ```CIT_ETHNICITY```

In [69]:
df['CIT_ETHNICITY'].unique()

array(['Non-Hispanic', 'Hispanic / Latino', 'Not Hispanic / Latino', <NA>,
       'Unknown', 'Hispanic', 'h'], dtype=object)

Like before, let's remove redundant values:

In [70]:
def fix_CIT_ETHNICITY(row):
  CIT_ETHNICITY = row['CIT_ETHNICITY']

  # Do nothing for NA values
  if pd.isna(CIT_ETHNICITY):
    return CIT_ETHNICITY

  if 'non-hispanic' in CIT_ETHNICITY.lower() or 'not hispanic' in CIT_ETHNICITY.lower():
    return 'Non-Hispanic'
  elif 'hispanic' in CIT_ETHNICITY.lower() or 'h' in CIT_ETHNICITY.lower():
    return 'Hispanic'
  elif 'unknown' in CIT_ETHNICITY.lower():
    return pd.NA

  return CIT_ETHNICITY

In [71]:
df['CIT_ETHNICITY'] = df.apply(fix_CIT_ETHNICITY, axis=1)

In [72]:
df['CIT_ETHNICITY'].unique()

array(['Non-Hispanic', 'Hispanic', <NA>], dtype=object)

## Cleaning ```SIMPLE_SUBJ_RE_GRP```

In [73]:
df['SIMPLE_SUBJ_RE_GRP'].unique()

array(['Black or African American', 'White', 'Other', 'Hispanic'],
      dtype=object)

Everything looks good here.

## Cleaning ```CITIZEN_CHARGE```

In [74]:
df['CITIZEN_CHARGE'].unique()

array(['None', 'Criminal Felony', 'Traffic Criminal',
       'Criminal Misdemeanor', 'Traffic Civil'], dtype=object)

Everything looks to be in order. For clarity, let's just replace ```None``` with ```No Charge``` so that no one wrongly assumes that ```None``` is an NA string.

In [75]:
df['CITIZEN_CHARGE'] = df['CITIZEN_CHARGE'].replace('None','No Charge')

In [76]:
df['CITIZEN_CHARGE'].unique()

array(['No Charge', 'Criminal Felony', 'Traffic Criminal',
       'Criminal Misdemeanor', 'Traffic Civil'], dtype=object)

## Cleaning ```HIGHEST_SHOW_FORCE```

In [77]:
df['HIGHEST_SHOW_FORCE'].unique()

array(['Gun', 'Impact Munitions', 'Irritants', 'Taser', 'None'],
      dtype=object)

Everything looks good.

## Cleaning ```HIGHEST_SHOW_FORCE```

In [78]:
df['SHOW_FORCE_COUNT'].unique()

array([1], dtype=int64)

No issues here. Let's convert the column to a numeric column (if it isn't already):

In [79]:
df['SHOW_FORCE_COUNT'] = pd.to_numeric(df['SHOW_FORCE_COUNT'])

# Normalization

Let's look at all of the columns again:

In [80]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4730 entries, 0 to 4729
Data columns (total 23 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   _id                 4730 non-null   int64         
 1   INC_IR_NO           4727 non-null   Int64         
 2   INC_IA_NO           4730 non-null   object        
 3   INC_DATE            4730 non-null   datetime64[ns]
 4   INC_YEAR            4730 non-null   int64         
 5   INC_TIME            4730 non-null   object        
 6   INC_DAY_WEEK        4730 non-null   object        
 7   INC_BEAT            4725 non-null   object        
 8   HUNDRED_BLOCK       4494 non-null   object        
 9   INC_CITY            4485 non-null   object        
 10  INC_STATE           4730 non-null   object        
 11  INC_ZIPCODE         4470 non-null   float64       
 12  INC_PRECINCT        4625 non-null   object        
 13  CIT_NUMBER          4728 non-null   float64     

Of course, we could just keep everything in a single table, but our the data cleaning hinted at an issue with this approach: Each incident can have multiple citizens involved, and each citizen can be involved in multiple incidents. In other words, there is a many-to-many relationship between citizens and incidents. Therefore, we'll need to split the data into several tables. We propose the following ERD:

![alt text](ERD.PNG)

Because of the many-to-many relationship, we need 3 tables:

* One to hold the incident reports
* One to hold the citizen information
* A junction table to connect the two

Let's break our dataframe into three separate dataframes to match this. Our primary keys will be ```INC_IR_NO``` for the incident reports table, ```CIT_NUMBER``` for the citizens table, and a new column we'll create later for the junction table. Before we split the dataframe, we'll have to remove all rows that have NA values in these columns, since we can't have null primary keys:

In [81]:
df.dropna(subset=['INC_IR_NO','CIT_NUMBER'],inplace=True)

For one last step, PostgreSQL (and the Python connector) prefer ```None``` rather than ```pd.NA``` for null values, so we'll need to adjust that before splitting the dataframe:

In [82]:
df.replace(pd.NA, None, inplace=True)
df.replace(np.nan, None, inplace=True)

# Save the completed, cleaned dataset
df.to_csv('data_clean.csv',index=False)

Now we can split the dataframe into the three tables:

In [83]:
incident_report_cols = ['INC_IR_NO','INC_IA_NO','INC_DATE','INC_YEAR','INC_TIME','INC_DAY_WEEK','INC_BEAT','HUNDRED_BLOCK',
                        'INC_CITY','INC_STATE','INC_ZIPCODE','INC_PRECINCT','HIGHEST_SHOW_FORCE','SHOW_FORCE_COUNT']
citizen_cols = ['CIT_NUMBER','CIT_GENDER','CIT_AGE','SUBJ_AGE_GROUP','CIT_RACE','CIT_ETHNICITY','SIMPLE_SUBJ_RE_GRP']
incident_details_cols = ['INC_IR_NO','CIT_NUMBER','CITIZEN_CHARGE']

df_incident_reports = df[incident_report_cols].drop_duplicates().reset_index(drop=True)
df_citizens = df[citizen_cols].drop_duplicates().reset_index(drop=True)
df_incident_details = df[incident_details_cols].drop_duplicates().reset_index(drop=True)

In [84]:
df_incident_reports.head()

,INC_IR_NO,INC_IA_NO,INC_DATE,INC_YEAR,INC_TIME,INC_DAY_WEEK,INC_BEAT,HUNDRED_BLOCK,INC_CITY,INC_STATE,INC_ZIPCODE,INC_PRECINCT,HIGHEST_SHOW_FORCE,SHOW_FORCE_COUNT
0,202500000255424,SOF25-0005,2025-02-18 23:11:00,2025,23:11:00,Tuesday,Maricopa,1XXXX West Rancho Santa Fe Boulevard,Avondale,AZ,85392.0,Out of City,Gun,1
1,202500000193276,SOF25-0006,2025-02-18 13:50:00,2025,13:50:00,Tuesday,721,6XXX West Mcdowell Road,Phoenix,AZ,85035.0,Maryvale/Estrella Precinct,Gun,1
2,202500000252010,SOF25-0007,2025-02-18 11:13:00,2025,11:13:00,Tuesday,731,2XXX West Heatherbrae Drive,Phoenix,AZ,85015.0,Mountain View Precinct,Gun,1
3,202500000252106,SOF25-0010,2025-02-18 12:44:00,2025,12:44:00,Tuesday,213,4XXX East Maya Way,Cave Creek,AZ,85331.0,Black Mountain Precinct,Gun,1
4,202500000189932,SOF25-0023,2025-02-18 12:45:00,2025,12:45:00,Tuesday,431,4XXX South 32nd Street,Phoenix,AZ,85040.0,South Mountain Precinct,Gun,1


In [85]:
df_incident_reports.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3556 entries, 0 to 3555
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   INC_IR_NO           3556 non-null   Int64         
 1   INC_IA_NO           3556 non-null   object        
 2   INC_DATE            3556 non-null   datetime64[ns]
 3   INC_YEAR            3556 non-null   int64         
 4   INC_TIME            3556 non-null   object        
 5   INC_DAY_WEEK        3556 non-null   object        
 6   INC_BEAT            3552 non-null   object        
 7   HUNDRED_BLOCK       3382 non-null   object        
 8   INC_CITY            3375 non-null   object        
 9   INC_STATE           3556 non-null   object        
 10  INC_ZIPCODE         3363 non-null   object        
 11  INC_PRECINCT        3477 non-null   object        
 12  HIGHEST_SHOW_FORCE  3556 non-null   object        
 13  SHOW_FORCE_COUNT    3556 non-null   int64       

In [86]:
df_citizens.head()

,CIT_NUMBER,CIT_GENDER,CIT_AGE,SUBJ_AGE_GROUP,CIT_RACE,CIT_ETHNICITY,SIMPLE_SUBJ_RE_GRP
0,44041.0,Male,30.0,30s,Black,Non-Hispanic,Black or African American
1,61390.0,Male,29.0,20s,White,Hispanic,White
2,61392.0,Male,45.0,40s,White,Non-Hispanic,White
3,61423.0,Male,53.0,50s,White,Hispanic,White
4,61465.0,Male,19.0,<20,Black,Non-Hispanic,Black or African American


In [87]:
df_citizens.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4544 entries, 0 to 4543
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   CIT_NUMBER          4544 non-null   float64
 1   CIT_GENDER          4241 non-null   object 
 2   CIT_AGE             4534 non-null   object 
 3   SUBJ_AGE_GROUP      4534 non-null   object 
 4   CIT_RACE            4131 non-null   object 
 5   CIT_ETHNICITY       4053 non-null   object 
 6   SIMPLE_SUBJ_RE_GRP  4544 non-null   object 
dtypes: float64(1), object(6)
memory usage: 248.6+ KB


In [88]:
df_incident_details.head()

,INC_IR_NO,CIT_NUMBER,CITIZEN_CHARGE
0,202500000255424,44041.0,No Charge
1,202500000193276,61390.0,No Charge
2,202500000252010,61392.0,No Charge
3,202500000252106,61423.0,Criminal Felony
4,202500000189932,61465.0,Criminal Felony


In [89]:
df_incident_details.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4715 entries, 0 to 4714
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   INC_IR_NO       4715 non-null   Int64  
 1   CIT_NUMBER      4715 non-null   float64
 2   CITIZEN_CHARGE  4715 non-null   object 
dtypes: Int64(1), float64(1), object(1)
memory usage: 115.2+ KB


Very good. Now we don't have redundant data that might cause issues in the future.

# Renaming Columns

Before we export to PostgreSQL, there's one final step. Let's go through each of the dataframes and rename the columns to be more human readable:

In [90]:
df_incident_reports.rename(columns={
    'INC_IR_NO':'incident_report_num',
    'INC_IA_NO':'show_of_force_report_num',
    'INC_DATE':'date',
    'INC_YEAR':'year',
    'INC_TIME':'time',
    'INC_DAY_WEEK':'day',
    'INC_BEAT':'beat',
    'HUNDRED_BLOCK':'address',
    'INC_CITY':'city',
    'INC_STATE':'state',
    'INC_ZIPCODE':'zipcode',
    'INC_PRECINCT':'precinct',
    'HIGHEST_SHOW_FORCE':'highest_show_of_force',
    'SHOW_FORCE_COUNT':'show_of_force_count'
},inplace=True)

df_incident_reports.head()

,incident_report_num,show_of_force_report_num,date,year,time,day,beat,address,city,state,zipcode,precinct,highest_show_of_force,show_of_force_count
0,202500000255424,SOF25-0005,2025-02-18 23:11:00,2025,23:11:00,Tuesday,Maricopa,1XXXX West Rancho Santa Fe Boulevard,Avondale,AZ,85392.0,Out of City,Gun,1
1,202500000193276,SOF25-0006,2025-02-18 13:50:00,2025,13:50:00,Tuesday,721,6XXX West Mcdowell Road,Phoenix,AZ,85035.0,Maryvale/Estrella Precinct,Gun,1
2,202500000252010,SOF25-0007,2025-02-18 11:13:00,2025,11:13:00,Tuesday,731,2XXX West Heatherbrae Drive,Phoenix,AZ,85015.0,Mountain View Precinct,Gun,1
3,202500000252106,SOF25-0010,2025-02-18 12:44:00,2025,12:44:00,Tuesday,213,4XXX East Maya Way,Cave Creek,AZ,85331.0,Black Mountain Precinct,Gun,1
4,202500000189932,SOF25-0023,2025-02-18 12:45:00,2025,12:45:00,Tuesday,431,4XXX South 32nd Street,Phoenix,AZ,85040.0,South Mountain Precinct,Gun,1


In [91]:
df_citizens.rename(columns={
    'CIT_NUMBER':'citizen_num',
    'CIT_GENDER':'gender',
    'CIT_AGE':'age',
    'SUBJ_AGE_GROUP':'age_group',
    'CIT_RACE':'race',
    'CIT_ETHNICITY':'ethnicity',
    'SIMPLE_SUBJ_RE_GRP':'race_ethnicity_group'
}, inplace=True)

df_citizens.head()

,citizen_num,gender,age,age_group,race,ethnicity,race_ethnicity_group
0,44041.0,Male,30.0,30s,Black,Non-Hispanic,Black or African American
1,61390.0,Male,29.0,20s,White,Hispanic,White
2,61392.0,Male,45.0,40s,White,Non-Hispanic,White
3,61423.0,Male,53.0,50s,White,Hispanic,White
4,61465.0,Male,19.0,<20,Black,Non-Hispanic,Black or African American


In [92]:
df_incident_details.rename(columns={
    'INC_IR_NO':'incident_report_num',
    'CIT_NUMBER':'citizen_num',
    'CITIZEN_CHARGE':'citizen_charge'
}, inplace=True)

# Add ID column
df_incident_details.insert(0,'incident_detail_num',df_incident_details.index + 1)

df_incident_details.head()

,incident_detail_num,incident_report_num,citizen_num,citizen_charge
0,1,202500000255424,44041.0,No Charge
1,2,202500000193276,61390.0,No Charge
2,3,202500000252010,61392.0,No Charge
3,4,202500000252106,61423.0,Criminal Felony
4,5,202500000189932,61465.0,Criminal Felony


# Exporting to PostgreSQL

Let's set up the connection parameters and the method to create the connection:

In [93]:
# Load .env file
load_dotenv()

DB_PARAMS = {
    "dbname": os.getenv("POSTGRES_DB"), 
    "user": os.getenv("POSTGRES_USER"),
    "password": os.getenv("POSTGRES_PASSWORD"),
    "host": os.getenv("POSTGRES_HOST"),
    "port": os.getenv("POSTGRES_PORT")
}

In [94]:
def get_db_connection():
    try:
        conn = psycopg2.connect(**DB_PARAMS)
        print("Database Connection Successful\n")
        return conn
    except Exception as e:
        print(f'Error Connecting to Database: {e}')

Now we can create the tables. We'll use a series of SQL queries that were automatically generated by pgAdmin4 from the earlier ERD:

In [95]:
conn = get_db_connection()
cursor = conn.cursor()

query = '''
BEGIN;

CREATE TABLE IF NOT EXISTS public.incident_reports
(
    incident_report_num bigint NOT NULL,
    show_of_force_report_num text,
    date date,
    year integer,
    "time" time without time zone,
    day text,
    beat text,
    address text,
    city text,
    state text,
    zipcode integer,
    precinct text,
    highest_show_of_force text,
    show_of_force_count integer,
    PRIMARY KEY (incident_report_num)
);

CREATE TABLE IF NOT EXISTS public.citizens
(
    citizen_num integer NOT NULL,
    gender text,
    age integer,
    age_group text,
    race text,
    ethnicity text,
    race_ethnicity_group text,
    PRIMARY KEY (citizen_num)
);

CREATE TABLE IF NOT EXISTS public.incident_details
(
    incident_detail_num integer NOT NULL,
    incident_report_num bigint NOT NULL,
    citizen_num integer NOT NULL,
    citizen_charge text,
    PRIMARY KEY (incident_detail_num)
);

ALTER TABLE IF EXISTS public.incident_details
    ADD FOREIGN KEY (incident_report_num)
    REFERENCES public.incident_reports (incident_report_num) MATCH SIMPLE
    ON UPDATE NO ACTION
    ON DELETE NO ACTION
    NOT VALID;


ALTER TABLE IF EXISTS public.incident_details
    ADD FOREIGN KEY (citizen_num)
    REFERENCES public.citizens (citizen_num) MATCH SIMPLE
    ON UPDATE NO ACTION
    ON DELETE NO ACTION
    NOT VALID;

END;
'''

cursor.execute(query)

print('Tables Created Successfully')

conn.commit()
cursor.close()
conn.close()

Database Connection Successful

Tables Created Successfully


Now we can begin inserting the values from the dataframes:

In [96]:
conn = get_db_connection()
cursor = conn.cursor()

print("Beginning insertion into incident reports table...")

# Loop over all citizens
for _, row in df_incident_reports.iterrows():
    # Insert current row - do nothing if the row already exists in the database
    cursor.execute('INSERT INTO incident_reports VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s) ON CONFLICT DO NOTHING;',
                (row['incident_report_num'],row['show_of_force_report_num'],row['date'],row['year'],row['time'],
                row['day'],row['beat'],row['address'],row['city'],row['state'],row['zipcode'],row['precinct'],
                row['highest_show_of_force'],row['show_of_force_count']))

print("Successfully inserted into incident reports table\n")

print("Beginning insertion into citizens table...")

# Loop over all citizens
for _, row in df_citizens.iterrows():
    # Insert current row - do nothing if the row already exists in the database
    cursor.execute('INSERT INTO citizens VALUES (%s, %s, %s, %s, %s, %s, %s) ON CONFLICT DO NOTHING;',
                (row['citizen_num'],row['gender'],row['age'],row['age_group'],row['race'],
                row['ethnicity'],row['race_ethnicity_group']))
    
print("Successfully inserted into citizens table\n")

print("Beginning insertion into incident_details table...")

# Loop over all incident details
for _, row in df_incident_details.iterrows():
    # Insert current row - do nothing if the row already exists in the database
    cursor.execute('INSERT INTO incident_details VALUES (%s, %s, %s, %s) ON CONFLICT DO NOTHING;',
                (row['incident_detail_num'],row['incident_report_num'],row['citizen_num'],row['citizen_charge']))
    
print("Successfully inserted into incident_details table\n")

print("All insertions successfully completed")

conn.commit()
cursor.close()
conn.close()

Database Connection Successful

Beginning insertion into incident reports table...
Successfully inserted into incident reports table

Beginning insertion into citizens table...
Successfully inserted into citizens table

Beginning insertion into incident_details table...
Successfully inserted into incident_details table

All insertions successfully completed
